# 02a (backup) — Training a flood segmentation model with ml4floods
*Backup of notebook 02a: the same training on a dataset prepared before the workshop (flood maps of Copernicus
EMS), in case the labels of Hands-on 1 are not usable.*

**"I want my own model."** The whole ml4floods training pipeline, one piece at a time, on a small regional
dataset around our common thread (the 2018 Thessaly flood):

1. The **WorldFloods format**: how an ML-ready flood dataset looks on disk.
2. The **split by flood event** (no data leakage).
3. The **configuration**: one JSON with everything ml4floods needs.
4. Large scenes → **tiles** of 256 × 256 px, computed live.
5. A **batch**: what the network actually sees (normalised tiles + labels).
6. The **model**: a linear model, a small CNN or a U-Net.
7. **Train** it and follow its **learning curves**.
8. **Evaluate** it on FARKADONA vs the classical methods of notebooks 01a/01b.
9. **Your turn**: change one thing and beat (or not) the instructor's model.

Nothing here is specific to floods: data in this format, tiling, batches, learning rate, splitting by event,
U-Nets… work for any segmentation problem in satellite images (crops, buildings, burnt areas…). ml4floods is
one of several open tools for this; see also TorchGeo, segmentation-models-pytorch or TerraTorch.

⏱ Training takes ~1 min on a GPU (Colab: *Runtime → Change runtime type → T4 GPU*) but about an hour on a
CPU. Without a GPU the notebook skips training and loads a model we trained before the workshop with the same
code and settings: everything after training works the same.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from course_utils import BANDS_S2, EVENT, IN_COLAB, ROOT, binary_metrics, find_event, read_event, save_fig

# Workaround for ml4floods 1.1.1 with recent PyTorch: the `verbose` argument of ReduceLROnPlateau was removed
class _ReduceLROnPlateau(torch.optim.lr_scheduler.ReduceLROnPlateau):
    def __init__(self, *args, verbose=None, **kwargs):
        super().__init__(*args, **kwargs)


torch.optim.lr_scheduler.ReduceLROnPlateau = _ReduceLROnPlateau

DATA_DIR = ROOT / "data" / "worldfloods_subset"
TRAIN = torch.cuda.is_available()   # no GPU: skip training and load the model trained before the workshop
DEVICE = "cuda" if TRAIN else "cpu"

## 1. The WorldFloods format
An ml4floods dataset is a folder per split (`train`, `val`, `test`) with a subfolder per **product**. Every
product of an event is a file with the **same name** in each subfolder:
- `S2/`: the Sentinel-2 image (13 bands, top-of-atmosphere reflectance × 10 000),
- `gt/`: the ground truth, on the same grid as the image,
- `PERMANENTWATERJRC/`: permanent water (JRC), `floodmaps/`: the original vector flood map,
- `meta/`: metadata of the event (dates, satellite of the flood map, pixel counts…).

Our data also have Sentinel-1 (`S1/`) and terrain (`HAND/`) for some events: same idea, more products.

In [ ]:
for split_name in ("train", "val", "test"):
    print(f"{split_name:5s}", sorted(p.name for p in (DATA_DIR / split_name).iterdir()))
sorted(p.name for p in (DATA_DIR / "train" / "S2").iterdir())

In [ ]:
EX = "EMSR429_AOI09_DEL_MONIT01"   # western Ireland, March 2020
meta = json.loads((DATA_DIR / "train" / "meta" / f"{EX}.json").read_text())
{k: meta[k] for k in ("event id", "event type", "satellite", "satellite date", "s2_date", "encoding_values")}

The flood map was drawn on a different image (`satellite`, `satellite date`; here a radar image) than the
Sentinel-2 image (`s2_date`): the closer the two dates, the better the label. The ground truth has **two layers**, encoded as `encoding_values` says:
layer 0 = invalid / clear / **cloud**, layer 1 = invalid / land / **water**.

In [ ]:
from georeader.rasterio_reader import RasterioReader

gt = RasterioReader(str(DATA_DIR / "train" / "gt" / f"{EX}.tif")).load()   # GeoTensor (2, y, x)
print(gt.shape, gt.crs)
for i, names in enumerate(meta["encoding_values"]):
    values, counts = np.unique(gt.values[i], return_counts=True)
    print(f"layer {i}:", {names[str(v)]: int(c) for v, c in zip(values, counts)})

Where does this ground truth come from? Copernicus EMS delivers flood maps as **vector layers** (polygons of
flooded areas, rivers, lakes…). ml4floods finds a Sentinel-2 image of the same days and **rasterises** the
polygons on its grid → a label for every pixel; clouds come from a cloud detector run on the image.

In [ ]:
import geopandas as gpd
from georeader import plot
from course_utils import COLORS_GT, GT_CLASSES, gt_classes
from matplotlib.patches import Patch

ex = read_event(*find_event(EX), EX)
floodmap = gpd.read_file(DATA_DIR / "train" / "floodmaps" / f"{EX}.geojson")
print(floodmap.groupby(["source", "w_class"]).size())

fig, axs = plt.subplots(1, 3, figsize=(20, 6.5))
rgb = (ex["s2"].isel({"band": [3, 2, 1]}) / 3000).clip(0, 1)   # B4, B3, B2 → [0, 1]: still a GeoTensor
plot.show(rgb, ax=axs[0], add_scalebar=True)
axs[0].set_title("Sentinel-2 (true colour)")
plot.show(rgb, ax=axs[1])
colors = {"flood": "tab:red", "hydro": "tab:blue", "hydro_l": "tab:blue"}
for source, col in colors.items():
    sel = floodmap[floodmap["source"] == source]
    if len(sel):
        sel.to_crs(ex["s2"].crs).plot(ax=axs[1], color=col, linewidth=1, alpha=0.7)
axs[1].legend(handles=[Patch(color="tab:red", label="flooded area (EMS)"),
                       Patch(color="tab:blue", label="rivers / lakes (EMS)")], loc="upper right")
axs[1].set_title("Copernicus EMS flood map (vector)")
plot.plot_segmentation_mask(gt_classes(gt), COLORS_GT, GT_CLASSES, ax=axs[2])   # the two layers in one map
axs[2].set_title("ML-ready ground truth (raster)")
plt.tight_layout()
save_fig(fig, "dl_cems_to_gt")

## 2. Split by flood event
Neighbouring pixels of the same flood look alike. If tiles of the same event end up in train and test,
the test score is too optimistic (**data leakage**). So we split by **event**:
- **train**: two areas of the Thessaly flood + six small floods from around the world (WorldFloods v2 train),
- **validation**: two floods with clean reference maps, Ylitornio (Finland) and Campogalliano (Italy), to
  monitor training and choose the decision threshold,
- **test**: FARKADONA (our common thread).

⚠ Why not another area of the Thessaly flood for validation? We tried SOFADES, next to FARKADONA, but its
reference map is raw Copernicus EMS: river lines are rasterised as water and the flooded fields visible in
Sentinel-2 are missing (even MNDWI > 0 gets IoU 0.21 there). **A reference map is not the truth**: check it
before you trust a metric computed on it.

In ml4floods the split is a JSON file listing the image and the label of every event:

In [ ]:
split = json.loads((DATA_DIR / "traintestval.json").read_text())
print(json.dumps(split["test"], indent=1))
events = pd.DataFrame([(s, os.path.basename(f)[:-4]) for s in split for f in split[s]["S2"]],
                      columns=["split", "event"])
events

In [ ]:
# No event in two splits (the check you should always do)
in_split = {s: set(events.event[events.split == s]) for s in split}
print("train ∩ val:", in_split["train"] & in_split["val"], "| train ∩ test:", in_split["train"] & in_split["test"],
      "| val ∩ test:", in_split["val"] & in_split["test"])

In [ ]:
# Different events can still be neighbours: where are the Thessaly areas? Each Sentinel-2 image at its real
# position, framed with the colour of its split. Train and test touch (same flood, same day) but do not overlap.
from georeader import read
from matplotlib.patches import Rectangle

SPLIT_COLORS = {"train": "tab:blue", "val": "tab:orange", "test": "tab:red"}
fig, ax = plt.subplots(figsize=(13, 7))
for i, (_, row) in enumerate(events[events.event.str.startswith("EMSR271")].iterrows()):
    reader = RasterioReader(str(DATA_DIR / row.split / "S2" / f"{row.event}.tif")).isel({"band": [3, 2, 1]})
    rgb = read.resize(reader.load(), resolution_dst=60)          # 10 m -> 60 m: enough for an overview map
    plot.show((rgb / 3000).clip(0, 1), ax=ax, add_scalebar=i == 0)
    xmin, ymin, xmax, ymax = rgb.bounds
    ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor=SPLIT_COLORS[row.split], lw=3))
    ax.text(xmin + 800, ymax - 800, row.event.split("_")[1][2:], va="top", fontsize=11, fontweight="bold",
            color="white", bbox=dict(facecolor=SPLIT_COLORS[row.split], edgecolor="none", alpha=0.85, pad=2))
ax.autoscale_view()
ax.legend(handles=[Patch(facecolor="none", edgecolor=SPLIT_COLORS[s], lw=3, label=s)
                   for s in SPLIT_COLORS if s in set(events[events.event.str.startswith("EMSR271")].split)],
          loc="lower right")
ax.set_title("EMSR271 Thessaly flood (Sentinel-2, 28 Feb 2018): areas and split")
save_fig(fig, "dl_split_map")

## 3. The configuration
Everything ml4floods needs is in **one configuration**: where the data are and how to read them
(`data_params`) and which model to train and how (`model_params`). We start from the WorldFloods v2 template
that comes with the package. Run the cell and read it.

In [ ]:
import ml4floods
from ml4floods.models.config_setup import get_default_config

template = os.path.join(os.path.dirname(ml4floods.__file__), "models/configurations/worldfloods_template_v2.json")
config = get_default_config(template)
config

It is a dictionary you can also read with dots (`config.data_params.batch_size`). We change what our small
dataset needs:

In [ ]:
from course_utils import ml4floods_split

split_file = ml4floods_split(DATA_DIR, windows=False)   # the split with absolute paths on this machine
config.data_params.path_to_splits = str(split_file.parent)
config.data_params.train_test_split_file = str(split_file)
config.data_params.channel_configuration = "bgriswirs"   # 6 bands: B2, B3, B4, B8, B11, B12 (as the model of 02b)
config.data_params.batch_size = 16                         # tiles per batch
config.data_params.num_workers = 2 if IN_COLAB else 0      # processes that read the tiles
config.data_params

In [ ]:
hp = config.model_params.hyperparameters
hp.channel_configuration = "bgriswirs"   # the model must know its input bands too
hp.num_channels = 6
hp.model_type = "unet"                   # "linear", "simplecnn" or "unet" (section 6)
hp.max_epochs = 5
# The template is tuned for long trainings on the whole WorldFloods (lr 1e-4, water errors weighted ×20).
# With only 5 epochs that makes the model call everything water → faster learning rate, milder weight.
hp.lr = 1e-3
hp.pos_weight = [1.0, 3.0]               # [cloud, water]
config.model_params.train, config.model_params.test = True, False   # train a new model (02b: test=True)
hp

## 4. Tiles, computed live
The scenes are thousands of pixels wide: we train on **tiles of 256 × 256 px** (`window_size`). ml4floods
drops the tiles that are mostly cloud or no data (`filter_windows`) and saves the list of tiles it keeps in a
JSON file. If the file exists it reuses it; if not, it computes it. We start from scratch:

In [ ]:
from ml4floods.models.dataset_setup import get_dataset

print("window_size:", config.data_params.window_size, "| filter_windows:", config.data_params.filter_windows)
windows_file = split_file.parent / "windows_traintestval.json"
windows_file.unlink(missing_ok=True)

dataset = get_dataset(config.data_params)   # a PyTorch Lightning DataModule: train / val / test tiles
print("windows file created:", windows_file.exists())

In [ ]:
windows = json.loads(windows_file.read_text())["slices"]
print(len(windows), "tiles kept. The first one:")
windows[0]

❓ `threshold_clouds` = maximum fraction of cloud / no data allowed in a tile. What happens to the number of
training tiles if we are stricter?

In [ ]:
for threshold in (0.8, 0.5, 0.2):
    config.data_params.filter_windows.threshold_clouds = threshold
    windows_file.unlink()
    n_train = len(get_dataset(config.data_params).train_dataset)
    print(f"\n>>> threshold_clouds = {threshold}: {n_train} training tiles\n")

config.data_params.filter_windows.threshold_clouds = 0.8   # back to the template value
windows_file.unlink()
dataset = get_dataset(config.data_params)

In [ ]:
# Tiles of one training scene: kept (green) and dropped because of clouds / no data (red)
from georeader.window_utils import window_bounds
from ml4floods.preprocess.tiling import WindowSize
from ml4floods.preprocess.utils import get_list_of_window_slices

TILED = "EMSR271_04STEFANOVIKIO_DEL_v1"
s2_file = str(DATA_DIR / next(f for f in split["train"]["S2"] if TILED in f))
kept = {(w.window.col_off, w.window.row_off) for w in dataset.train_dataset.list_of_windows if TILED in w.file_name}
all_windows = get_list_of_window_slices([s2_file], window_size=WindowSize(256, 256))

reader = RasterioReader(s2_file)
fig, ax = plt.subplots(figsize=(10, 8.5))
plot.show((reader.isel({"band": [3, 2, 1]}).load() / 3000).clip(0, 1), ax=ax)
for w in all_windows:
    ok = (w.window.col_off, w.window.row_off) in kept
    xmin, ymin, xmax, ymax = window_bounds(w.window, reader.transform)   # pixels → map coordinates
    ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor="lime" if ok else "red",
                           lw=1.2))
ax.set_title(f"{TILED}: {len(kept)} of {len(all_windows)} tiles of 256×256 px used for training")
ax.set_axis_off()
save_fig(fig, "dl_tiling")

## 5. A batch: what the network sees
The DataModule reads the tiles, keeps the bands of `channel_configuration`, **normalises** them and groups them
in **batches**. Let's take one batch.

In [ ]:
batch = next(iter(dataset.train_dataloader()))
{k: (tuple(v.shape), v.dtype) for k, v in batch.items()}

`image`: 16 tiles × 6 bands × 256 × 256 px. `mask`: the two layers of the ground truth (cloud, water) for
each tile. Each band $b$ is **standardised** (z-score) with the mean $\mu_b$ and standard deviation $\sigma_b$
of that band over the whole WorldFloods dataset:

$$z_b = \frac{x_b - \mu_b}{\sigma_b}$$

→ every band is centred at 0 with a spread of ~1, whatever its original range (the network learns faster and
no band dominates just because its numbers are bigger):

In [ ]:
from ml4floods.models.model_setup import get_channel_configuration_bands
from ml4floods.preprocess.worldfloods.normalize import get_normalisation

bands = [BANDS_S2[i] for i in get_channel_configuration_bands(config.data_params.channel_configuration)]
mean, std = get_normalisation(config.data_params.channel_configuration)
pd.DataFrame({"band": bands, "mean": mean.ravel(), "std": std.ravel()}).round(0)

In [ ]:
x = batch["image"].numpy()
raw = x * std.reshape(1, -1, 1, 1) + mean.reshape(1, -1, 1, 1)   # undo the normalisation
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
for i, b in enumerate(bands):
    axs[0].hist(raw[:, i].ravel(), bins=100, histtype="step", label=b)
    axs[1].hist(x[:, i].ravel(), bins=100, histtype="step", label=b)
axs[0].set(title="as stored (reflectance × 10 000)", xlim=(0, 6000))
axs[1].set(title="as the network sees it (normalised)", xlim=(-3, 4))
axs[1].legend()
plt.show()

In [ ]:
# Tiles of a batch are plain arrays (no georeference) → matplotlib's imshow, with the colours of COLORS_GT
rgb_bands = [bands.index(b) for b in ("B4", "B3", "B2")]
fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axs[0, i].imshow((raw[i, rgb_bands] / 3000).clip(0, 1).transpose(1, 2, 0))   # (band, y, x) → (y, x, band)
    axs[1, i].imshow(COLORS_GT[gt_classes(batch["mask"][i].numpy())])
for ax in axs.ravel():
    ax.set_axis_off()
axs[0, 0].set_title("image (true colour)", loc="left")
axs[1, 0].set_title("mask (land / water / cloud)", loc="left")
plt.show()

## 6. The model
`get_model` builds the network named in `model_type` and wraps it with its loss and optimiser. ml4floods has:
- `linear`: a 1 × 1 convolution = a **linear model per pixel** (it only sees the bands of each pixel, like a
  water index with learned weights),
- `simplecnn`: a few 3 × 3 convolutions (sees a small neighbourhood),
- `unet`: the U-Net of the slides (sees context at several scales).

In [ ]:
from ml4floods.models.worldfloods_model import configure_architecture

for model_type in ("linear", "simplecnn", "unet"):
    net = configure_architecture({"model_type": model_type, "num_channels": 6, "num_classes": 2})
    print(f"{model_type:10s} {sum(p.numel() for p in net.parameters()):>10,} parameters")
configure_architecture({"model_type": "linear", "num_channels": 6, "num_classes": 2})

Two outputs per pixel (`num_classes = 2`): one for **cloud** and one for **water**, each turned into a
probability by a sigmoid. Before training, the weights are random. Its prediction on four tiles of our batch:

In [ ]:
from ml4floods.models.model_setup import get_model

torch.manual_seed(config.seed)
model = get_model(config.model_params)   # needs model_params.train = True (test = True loads saved weights)
tiles = batch["image"][:4]


def show_water(model, title):
    model.eval()   # prediction mode (e.g. batch normalisation uses its stored statistics)
    with torch.no_grad():
        p_water = torch.sigmoid(model(tiles.to(model.device)))[:, 1].cpu().numpy()
    model.train()  # back to training mode, otherwise Lightning would train it in prediction mode
    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    for i, ax in enumerate(axs):
        ax.imshow(p_water[i], cmap="Blues", vmin=0, vmax=1)
        ax.set_axis_off()
    axs[0].set_title(title, loc="left")
    plt.show()


show_water(model, "P(water) of the untrained model")

## 7. Train
One **epoch** = the model sees every training tile once, in **batches** of 16 tiles. After each batch the
error (loss) is back-propagated and the weights are updated (stochastic gradient descent). After each
epoch the model is checked on the **validation** events, which it never trains on.

In [ ]:
from course_utils import learning_curve_callback
from pytorch_lightning import Trainer

EXPERIMENT = "my_unet"
model_dir = ROOT / "models" / EXPERIMENT

if TRAIN:
    trainer = Trainer(max_epochs=hp.max_epochs, accelerator="auto", logger=False, enable_checkpointing=False,
                      callbacks=[learning_curve_callback(model_dir / "learning_curve.csv")])
    t0 = time.time()
    trainer.fit(model, dataset)
    print(f"training time: {(time.time() - t0) / 60:.1f} min")

    # Save as the models on HuggingFace: <folder>/config.json + <folder>/model.pt
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_dir / "model.pt")
    (model_dir / "config.json").write_text(json.dumps(config, indent=2))
else:
    EXPERIMENT = "course_unet"   # trained before the workshop with this notebook (same settings)
    model_dir = ROOT / "models" / EXPERIMENT
    print(f"No GPU: training skipped, using {EXPERIMENT} (trained before the workshop with the same settings)")
    saved = get_default_config(str(model_dir / "config.json"))
    saved.model_params.model_folder, saved.model_params.test = str(model_dir.parent), True
    model = get_model(saved.model_params, EXPERIMENT)   # test mode: loads <model_folder>/<EXPERIMENT>/model.pt

show_water(model, "P(water) after training")

### Learning curves
**Training loss**: error on the tiles the model learns from. **Validation loss**: error on floods it never
sees (one point per epoch). The first one always goes down; the second one tells us if the model
**generalises**.

In [ ]:
from course_utils import plot_learning_curve

plt.figure(figsize=(9, 5))
plot_learning_curve(model_dir / "learning_curve.csv", title=f"{EXPERIMENT}: learning curves",
                    val_label="validation loss (Ylitornio + Campogalliano)")
plt.show()

❓ What happens if we keep training for longer? Before the workshop we trained the same model for 15 epochs
(its epoch 5 is the model loaded when there is no GPU).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_learning_curve(ROOT / "models" / "course_unet_curve" / "learning_curve.csv", ax=ax,
                    title="Same model, 15 epochs: training vs validation loss",
                    val_label="validation loss (Ylitornio + Campogalliano)")
ax.axvline(5, color="gray", ls="--"); ax.text(5.2, ax.get_ylim()[1] * 0.95, "today: 5 epochs", va="top", color="gray")
save_fig(fig, "dl_learning_curve")

## 8. Evaluate
To use the model on a whole scene we load it as ml4floods saves it (`config.json` + `model.pt`):
`load_inference_function` normalises the image, cuts it in tiles, runs the network and stitches the output.
It works with PyTorch tensors: numpy → torch on the way in; on the way out `array_as_geotensor` puts the
georeference back (PyTorch does not know about it).

In [ ]:
from georeader.geotensor import GeoTensor
from ml4floods.scripts.inference import load_inference_function


def run_model(model_dir, s2: GeoTensor) -> tuple[GeoTensor, GeoTensor]:
    """P(water) and cloud mask of a model saved by ml4floods on a 13-band Sentinel-2 image."""
    inference_fn, model_config = load_inference_function(str(model_dir), device_name=DEVICE, max_tile_size=1024)
    channels = get_channel_configuration_bands(model_config.data_params.channel_configuration)   # its bands
    pred_class, prob = inference_fn(torch.tensor(s2.values[channels]))   # {0 invalid, 1 land, 2 water, 3 cloud}
    return s2.array_as_geotensor(prob[1].numpy()), s2.array_as_geotensor(pred_class.numpy() == 3)


d = read_event(*find_event(EVENT), EVENT)   # FARKADONA, the test event
prob, cloud = run_model(model_dir, d["s2"])
gt = d["gt"]
ref, valid = gt[1] == 2, (gt[1] != 0) & (gt[0] != 2)   # reference water; labelled pixels outside the clouds
print(type(prob).__name__, "| same grid as the image:", prob.same_extent(d["s2"]))

The model gives a probability of water for every pixel; the map depends on the **threshold** we choose. We
choose it on the **validation** events and only then look at the test event (choosing it on the test event
would be leakage too). We put the pixels of the validation floods together and keep the threshold with the
best IoU:

In [ ]:
VAL = [read_event(*find_event(ev), ev) for ev in (os.path.basename(f)[:-4] for f in split["val"]["S2"])]
ref_val = np.concatenate([(v["gt"][1] == 2).values.ravel() for v in VAL])
valid_val = np.concatenate([((v["gt"][1] != 0) & (v["gt"][0] != 2)).values.ravel() for v in VAL])
thresholds = np.round(np.arange(0.1, 0.95, 0.05), 2)


def choose_threshold(model_dir) -> tuple[float, list]:
    """Threshold on P(water) with the best IoU on the validation floods (+ the IoU of every threshold)."""
    outputs = [run_model(model_dir, v["s2"]) for v in VAL]
    prob_val = np.concatenate([p.values.ravel() for p, _ in outputs])
    cloud_val = np.concatenate([c.values.ravel() for _, c in outputs])
    iou = [binary_metrics((prob_val > th) & ~cloud_val, ref_val, valid_val)["IoU"] for th in thresholds]
    return float(thresholds[np.argmax(iou)]), iou


TH_WATER, iou_val = choose_threshold(model_dir)
print(f"best threshold on the validation floods: {TH_WATER} (IoU {max(iou_val):.2f})")

In [ ]:
iou_test = [binary_metrics((prob > th) & ~cloud, ref, valid)["IoU"] for th in thresholds]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, iou_val, "o-", color="tab:orange", label="validation (Ylitornio + Campogalliano)")
ax.plot(thresholds, iou_test, "o-", color="tab:red", label=f"test ({EVENT.split('_')[1]})")
ax.axvline(TH_WATER, color="k", ls="--", label=f"chosen on validation: {TH_WATER}")
ax.set(xlabel="threshold on P(water)", ylabel="IoU", ylim=(0, 1), title="Choosing the threshold")
ax.grid(alpha=0.3); ax.legend()
save_fig(fig, "dl_threshold")

### Deep learning vs the classical methods (FARKADONA)
❓ **Before running:** which method will have the best IoU? Where will our U-Net still make mistakes?

Same pixels for all methods: valid and outside the clouds of the reference map.

In [ ]:
from course_utils import edge_otsu, mndwi   # from notebooks 01a and 01b

vv = d["s1"][0]
t_edge, _ = edge_otsu(vv, t0=-16.0)
OURS = f"our U-Net, P(water) > {TH_WATER} (deep learning)"
methods = {   # every flood map is a GeoTensor on the same grid
    "MNDWI > 0 (optical)": mndwi(d["s2"]) > 0,
    "Edge Otsu VV + HAND < 15 m (SAR)": (vv < t_edge) & (d["hand"] < 15),
    OURS: (prob > TH_WATER) & ~cloud,
}
table = pd.DataFrame({name: binary_metrics(p, ref, valid) for name, p in methods.items()}).T.round(2)
table

In [ ]:
from course_utils import COLORS_CONFUSION, CONFUSION_CLASSES, confusion_classes


def confusion_panels(axs, items):
    for i, (ax, (name, p)) in enumerate(zip(axs, items)):
        m = table.loc[name]
        plot.plot_segmentation_mask(confusion_classes(p, ref, valid), COLORS_CONFUSION, CONFUSION_CLASSES, ax=ax,
                                    legend=i == 0)
        ax.set_title(f"{name}\nP {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}", fontsize=12)
        ax.set_axis_off()


fig, axs = plt.subplots(1, 2, figsize=(14, 6.5))     # classical only (slide "Where rules break")
confusion_panels(axs, list(methods.items())[:2])
plt.tight_layout()
save_fig(fig, "dl_rules_break")
plt.close(fig)

fig, axs = plt.subplots(1, 3, figsize=(20, 6.5))
confusion_panels(axs, methods.items())
plt.tight_layout()
save_fig(fig, "dl_vs_classical")

### Where does our model fail?
Zoom on its largest false alarm.

In [ ]:
from skimage.measure import label, regionprops

fp = label((methods[OURS] & ~ref & valid).values)   # skimage: plain numpy
r0, c0, r1, c1 = max(regionprops(fp), key=lambda r: r.area).bbox
pad = 150
rows, cols = slice(max(r0 - pad, 0), r1 + pad), slice(max(c0 - pad, 0), c1 + pad)
zoom = d["s2"][:, rows, cols]   # slicing a GeoTensor keeps the georeference
fig, axs = plt.subplots(1, 3, figsize=(18, 6.5))
plot.show((zoom.isel({"band": [3, 2, 1]}) / 3000).clip(0, 1), ax=axs[0], add_scalebar=True)
axs[0].set_title("Sentinel-2 true colour")
plot.show((zoom.isel({"band": [11, 7, 3]}) / 4500).clip(0, 1), ax=axs[1])   # B11, B8, B4
axs[1].set_title("SWIR / NIR / Red")
plot.plot_segmentation_mask(confusion_classes(methods[OURS], ref, valid)[rows, cols], COLORS_CONFUSION,
                            CONFUSION_CLASSES, ax=axs[2])
axs[2].set_title("our U-Net: errors")
for ax in axs:
    ax.set_axis_off()
plt.tight_layout()
save_fig(fig, "dl_ours_failure")

Dark in every band, next to a cloud: a **cloud shadow**. Our 8 training floods barely contain cloud shadows,
so the model never learned that they are not water → **a model fails on what it has not seen**. The global
model of notebook 02b was trained on ~500 flood maps from all over the world.

## 9. Your turn: change one thing
Change **one** setting in `MY_CHANGE` and run the cell: it trains a new model with the configuration of
section 3 + your change, chooses its threshold on validation and compares it on FARKADONA with the
instructor's U-Net. Does it get better or worse? Why? Some ideas:
- `{"model_type": "linear"}`: no spatial context at all, only the bands of each pixel. How far does it get?
- `{"model_type": "simplecnn"}`: a little context.
- `{"lr": 1e-1}` or `{"lr": 1e-5}`: a crazy-high or a tiny learning rate.
- `{"channel_configuration": "rgb"}`: only the visible bands (a cheap camera), or `"all"` (13 bands).
- `{"max_epochs": 2}` or `{"max_epochs": 10}`, `{"pos_weight": [1.0, 20.0]}` (the template's water weight).

In [ ]:
from ml4floods.models.utils.configuration import AttrDict

MY_CHANGE = {"model_type": "linear"}   # ← your change


def train_variant(change: dict, name: str = "my_variant"):
    cfg = AttrDict.from_nested_dicts(json.loads(json.dumps(config)))   # a copy of the configuration
    h = cfg.model_params.hyperparameters
    for k, v in change.items():
        if k == "channel_configuration":   # the data and the model must agree on the bands
            cfg.data_params.channel_configuration = h.channel_configuration = v
            h.num_channels = len(get_channel_configuration_bands(v))
        else:
            h[k] = v
    variant_dir = ROOT / "models" / name
    torch.manual_seed(cfg.seed)
    m = get_model(cfg.model_params)
    t0 = time.time()
    Trainer(max_epochs=h.max_epochs, accelerator="auto", logger=False, enable_checkpointing=False,
            callbacks=[learning_curve_callback(variant_dir / "learning_curve.csv")]).fit(m, get_dataset(cfg.data_params))
    print(f"training time: {(time.time() - t0) / 60:.1f} min")
    variant_dir.mkdir(parents=True, exist_ok=True)
    torch.save(m.state_dict(), variant_dir / "model.pt")
    (variant_dir / "config.json").write_text(json.dumps(cfg, indent=2))
    return variant_dir


if not TRAIN:
    print("Training needs a GPU (Runtime → Change runtime type → T4 GPU).")
else:
    variant_dir = train_variant(MY_CHANGE)
    th, _ = choose_threshold(variant_dir)
    p, c = run_model(variant_dir, d["s2"])
    mine = pd.Series(binary_metrics((p > th) & ~c, ref, valid)).round(2)
    print(pd.DataFrame({"instructor's U-Net": table.loc[OURS], f"you: {MY_CHANGE} (threshold {th})": mine}))

**Take-aways**
- ml4floods turns flood maps made by others into ML-ready data, and a configuration into a trained model, in a
  few lines: format → split → config → tiles → batches → model → training → evaluation.
- Split by event; tile large scenes; evaluate on floods the model has never seen.
- A small model trained in minutes on a few floods already beats the rules near its training region, but it
  fails on what it has not seen (cloud shadows). What would you need to use it on the next flood in another
  continent? → notebook 02b.